# Value Score Engineering

The goal of this notebook is to create position-adjusted EPA-based offensive value scores for NFL quarterbacks, running backs, wide receivers, and tight ends.

Player value is difficult to measure directly, so this project uses EPA (*Expected Points Added*) as the core production metric. EPA is useful because it captures the expected point impact of a player's plays rather than only counting raw box-score production.

This notebook now creates two related EPA-based scores:

1. `value_score_total_epa`: position-adjusted total season EPA. This is the better primary metric for season-level player value and future salary-efficiency analysis.
2. `value_score_per_game`: position-adjusted EPA per game. This is useful for measuring how productive a player was when active, but it can inflate shorter seasons.

The primary `value_score` is set equal to `value_score_total_epa`. For quarterbacks, EPA uses passing plus rushing EPA. For running backs, wide receivers, and tight ends, EPA uses rushing plus receiving EPA. Scores are standardized within each season-position group so that players are compared to others at the same position in the same season.


## Load Cleaned Player-Season Data

This section loads the cleaned player-season dataset created in the previous notebook. Each row represents one player-season for a quarterback, running back, wide receiver, or tight end.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

def find_project_root(expected_file):
    """Find the repo root from common VS Code/Jupyter working directories."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / expected_file).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {expected_file} from working directory {Path.cwd()}"
    )

project_root = find_project_root("data/processed/skill_player_seasons_2016_2025.csv")
processed_dir = project_root / "data" / "processed"

skill_season_path = processed_dir / "skill_player_seasons_2016_2025.csv"
skill_season = pd.read_csv(skill_season_path)

print(skill_season.shape)
skill_season.head()


(6082, 69)


,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,birth_date,height,weight,years_exp,entry_year,rookie_year,draft_club,draft_number,college,age
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,1977-08-03,76.0,225.0,16.0,2000.0,2000.0,NE,NaN,Michigan,39
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,1979-05-12,69.0,195.0,NaN,NaN,NaN,NaN,NaN,Utah,37
2,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,1979-01-15,72.0,209.0,15.0,2001.0,2001.0,SD,NaN,Purdue,37
3,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,1980-01-09,75.0,230.0,14.0,2002.0,2002.0,NaN,NaN,Maryland,36
4,2016,00-0021206,J.McCown,Josh McCown,QB,CLE,90,165,1100,6,...,1979-07-04,76.0,218.0,14.0,2002.0,2002.0,ARI,NaN,Sam Houston State,37


## Create Final Supporting Features

The main value score will be based on EPA per game, but I also create several supporting features that will be useful for interpreting player rankings later.

For non-quarterbacks, touchdowns per game help describe scoring production. For quarterbacks, interceptions per game help describe negative plays. These variables are not the main value score, but they will be useful in later analysis.

In [2]:
# Keep only players with at least one game played
skill_season = skill_season[skill_season["games_played"] > 0].copy()

# Non-QB scoring feature
skill_season["scrimmage_tds_per_game"] = (
    skill_season["scrimmage_tds"] / skill_season["games_played"]
)

# QB negative-play feature
skill_season["interceptions_per_game"] = (
    skill_season["passing_interceptions"] / skill_season["games_played"]
)

# EPA-only benchmark score variable for all players
skill_season["epa_only_score"] = skill_season["epa_per_game"]

skill_season.head()


,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,years_exp,entry_year,rookie_year,draft_club,draft_number,college,age,scrimmage_tds_per_game,interceptions_per_game,epa_only_score
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,16.0,2000.0,2000.0,NE,NaN,Michigan,39,0.000000,0.166667,11.719361
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,Utah,37,0.357143,0.000000,2.710512
2,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,15.0,2001.0,2001.0,SD,NaN,Purdue,37,0.125000,0.937500,6.604510
3,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,14.0,2002.0,2002.0,NaN,NaN,Maryland,36,0.000000,0.000000,1.420066
4,2016,00-0021206,J.McCown,Josh McCown,QB,CLE,90,165,1100,6,...,14.0,2002.0,2002.0,ARI,NaN,Sam Houston State,37,0.000000,1.200000,-5.810295


## Filter Out Small-Sample Players

Players with fewer than four games played were removed from the value score dataset to reduce noise from very small samples.

In [3]:
# Filter out very small sample players
value_df = skill_season[skill_season["games_played"] >= 4].copy()

print(value_df.shape)
value_df.head()

(4796, 72)


,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,years_exp,entry_year,rookie_year,draft_club,draft_number,college,age,scrimmage_tds_per_game,interceptions_per_game,epa_only_score
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,16.0,2000.0,2000.0,NE,NaN,Michigan,39,0.000000,0.166667,11.719361
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,Utah,37,0.357143,0.000000,2.710512
2,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,15.0,2001.0,2001.0,SD,NaN,Purdue,37,0.125000,0.937500,6.604510
4,2016,00-0021206,J.McCown,Josh McCown,QB,CLE,90,165,1100,6,...,14.0,2002.0,2002.0,ARI,NaN,Sam Houston State,37,0.000000,1.200000,-5.810295
5,2016,00-0021429,C.Palmer,Carson Palmer,QB,ARI,364,597,4233,26,...,13.0,2003.0,2003.0,CIN,NaN,USC,37,0.000000,0.933333,1.750755


## Define a Group Z-Score Function

The value score should compare players to others at the same position in the same season. This is important because raw EPA differs across positions, especially between quarterbacks and non-quarterbacks.

This function creates z-scores within season-position groups. A z-score of 2 means the player was about two standard deviations above the average player at that position in that season.

In [4]:
def add_group_zscore(df, cols, group_cols=["season", "position"]):
    """
    Add z-score columns for selected variables within season-position groups.
    This compares each player to others at the same position in the same season.
    """
    df = df.copy()
    
    for col in cols:
        z_col = f"{col}_z"
        group_mean = df.groupby(group_cols)[col].transform("mean")
        group_std = df.groupby(group_cols)[col].transform("std")
        
        df[z_col] = (df[col] - group_mean) / group_std
    
    return df

## Create EPA-Based Value Scores

This section creates both total-EPA and per-game EPA value scores.

For quarterbacks, I use:

- `qb_epa` for total season value
- `qb_epa_per_game` for per-game value

For running backs, wide receivers, and tight ends, I use:

- `scrimmage_epa` for total season value
- `scrimmage_epa_per_game` for per-game value

The main `value_score` is set to the total-EPA z-score because salary-efficiency analysis is ultimately about season-level production relative to cost. The per-game score is retained as a useful secondary measure.


In [5]:
# Split QBs and non-QBs
non_qb_positions = ["RB", "WR", "TE"]

non_qb = value_df[value_df["position"].isin(non_qb_positions)].copy()
qbs = value_df[value_df["position"] == "QB"].copy()

# Non-QB EPA value scores
non_qb = non_qb.dropna(subset=["scrimmage_epa", "scrimmage_epa_per_game"])
non_qb = add_group_zscore(non_qb, ["scrimmage_epa", "scrimmage_epa_per_game"])

non_qb["value_epa_total"] = non_qb["scrimmage_epa"]
non_qb["value_epa_per_game"] = non_qb["scrimmage_epa_per_game"]
non_qb["value_score_total_epa"] = non_qb["scrimmage_epa_z"]
non_qb["value_score_per_game"] = non_qb["scrimmage_epa_per_game_z"]
non_qb["value_score"] = non_qb["value_score_total_epa"]
non_qb["value_metric"] = "position_adjusted_total_epa"

# QB EPA value scores
qbs = qbs.dropna(subset=["qb_epa", "qb_epa_per_game"])
qbs = add_group_zscore(qbs, ["qb_epa", "qb_epa_per_game"])

qbs["value_epa_total"] = qbs["qb_epa"]
qbs["value_epa_per_game"] = qbs["qb_epa_per_game"]
qbs["value_score_total_epa"] = qbs["qb_epa_z"]
qbs["value_score_per_game"] = qbs["qb_epa_per_game_z"]
qbs["value_score"] = qbs["value_score_total_epa"]
qbs["value_metric"] = "position_adjusted_total_epa"


## Combine Quarterbacks and Non-Quarterbacks

After creating separate EPA-based value scores for quarterbacks and non-quarterbacks, I combine the datasets back into one value-scored player-season dataset.

In [6]:
value_scored = pd.concat([qbs, non_qb], ignore_index=True)

print(value_scored.shape)
value_scored.head()

(4796, 82)


,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,qb_epa_z,qb_epa_per_game_z,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,value_metric,scrimmage_epa_z,scrimmage_epa_per_game_z
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,2.127596,2.264787,140.632333,11.719361,2.127596,2.264787,2.127596,position_adjusted_total_epa,NaN,NaN
1,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,1.491620,1.172466,105.672159,6.604510,1.491620,1.172466,1.491620,position_adjusted_total_epa,NaN,NaN
2,2016,00-0021206,J.McCown,Josh McCown,QB,CLE,90,165,1100,6,...,-0.959196,-1.478825,-29.051473,-5.810295,-0.959196,-1.478825,-0.959196,position_adjusted_total_epa,NaN,NaN
3,2016,00-0021429,C.Palmer,Carson Palmer,QB,ARI,364,597,4233,26,...,0.047023,0.135903,26.261320,1.750755,0.047023,0.135903,0.047023,position_adjusted_total_epa,NaN,NaN
4,2016,00-0022803,E.Manning,Eli Manning,QB,NYG,377,598,4027,26,...,-0.798394,-0.507764,-20.212029,-1.263252,-0.798394,-0.507764,-0.798394,position_adjusted_total_epa,NaN,NaN


## Add Position-Season Ranks and Percentiles

Ranks and percentiles are calculated from the primary `value_score`, which is currently the position-adjusted total-EPA score.

The rank compares players within the same season and position. The percentile gives a relative standing, where values closer to 1 indicate stronger full-season production within that season-position group.


In [7]:
value_scored["position_season_rank"] = (
    value_scored
    .groupby(["season", "position"])["value_score"]
    .rank(ascending=False, method="min")
)

value_scored["position_season_percentile"] = (
    value_scored
    .groupby(["season", "position"])["value_score"]
    .rank(pct=True)
)

value_scored[
    ["season", "player_display_name", "position", "team", "games_played",
     "value_epa_total", "value_epa_per_game", "value_score_total_epa",
     "value_score_per_game", "value_score", "position_season_rank",
     "position_season_percentile"]
].sort_values(["season", "position", "position_season_rank"]).head(30)


,season,player_display_name,position,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
15,2016,Matt Ryan,QB,ATL,16,166.575950,10.410997,2.599548,1.985374,2.599548,1.0,1.00
8,2016,Aaron Rodgers,QB,GB,16,150.577515,9.411095,2.308513,1.771836,2.308513,2.0,0.98
0,2016,Tom Brady,QB,NE,12,140.632333,11.719361,2.127596,2.264787,2.127596,3.0,0.96
47,2016,Dak Prescott,QB,DAL,16,138.916125,8.682258,2.096376,1.616187,2.096376,4.0,0.94
29,2016,Kirk Cousins,QB,WAS,16,110.445666,6.902854,1.578457,1.236180,1.578457,5.0,0.92
1,2016,Drew Brees,QB,NO,16,105.672159,6.604510,1.491620,1.172466,1.491620,6.0,0.90
17,2016,Matthew Stafford,QB,DET,16,97.534789,6.095924,1.343590,1.063853,1.343590,7.0,0.88
5,2016,Ben Roethlisberger,QB,PIT,14,79.763750,5.697411,1.020309,0.978746,1.020309,8.0,0.86
31,2016,Andrew Luck,QB,IND,15,67.054950,4.470330,0.789118,0.716693,0.789118,9.0,0.84
23,2016,Andy Dalton,QB,CIN,16,63.623157,3.976447,0.726688,0.611220,0.726688,10.0,0.82


## Sanity Check Player Rankings

To make sure the value score is reasonable, I check the top players at each position for a recent season.

The table includes both total-EPA value and per-game value. This helps separate players who produced the most over the season from players who were highly productive on a per-game basis.


In [8]:
season_to_check = 2024

for pos in ["QB", "RB", "WR", "TE"]:
    print()
    print("Top", pos + "s", "in", season_to_check)
    display(
        value_scored[
            (value_scored["season"] == season_to_check) &
            (value_scored["position"] == pos)
        ][
            ["player_display_name", "team", "games_played",
             "value_epa_total", "value_epa_per_game",
             "value_score_total_epa", "value_score_per_game",
             "value_score", "position_season_rank",
             "position_season_percentile"]
        ]
        .sort_values("value_score", ascending=False)
        .head(10)
    )



Top QBs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
451,Josh Allen,BUF,16,187.113574,11.694598,2.753053,2.341023,2.753053,1.0,1.000000
449,Lamar Jackson,BAL,17,183.931545,10.819503,2.701035,2.157571,2.701035,2.0,0.982759
441,Jared Goff,DET,17,165.429827,9.731166,2.398578,1.929417,2.398578,3.0,0.965517
462,Joe Burrow,CIN,17,131.179774,7.716457,1.838674,1.507061,1.838674,4.0,0.948276
450,Baker Mayfield,TB,17,122.182661,7.187215,1.691593,1.396113,1.691593,5.0,0.931034
484,Jayden Daniels,WAS,17,119.373123,7.021948,1.645664,1.361467,1.645664,6.0,0.913793
468,Brock Purdy,SF,15,97.007326,6.467155,1.280039,1.245162,1.280039,7.0,0.896552
447,Patrick Mahomes,KC,16,95.446626,5.965414,1.254526,1.139979,1.254526,8.0,0.879310
458,Tua Tagovailoa,MIA,11,86.305197,7.845927,1.105086,1.534202,1.105086,9.0,0.862069
453,Kyler Murray,ARI,17,83.587614,4.916918,1.060660,0.920176,1.060660,10.0,0.844828



Top RBs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
4282,Jahmyr Gibbs,DET,17,50.032154,2.943068,3.307019,2.716515,3.307019,1.0,1.000000
3955,Derrick Henry,BAL,17,49.606434,2.918026,3.281219,2.696549,3.281219,2.0,0.991453
4026,Saquon Barkley,PHI,16,49.461172,3.091323,3.272415,2.834717,3.272415,3.0,0.982906
4295,Bucky Irving,TB,17,35.450351,2.085315,2.423312,2.032642,2.423312,4.0,0.974359
4220,Bijan Robinson,ATL,17,25.584958,1.504998,1.825435,1.569964,1.825435,5.0,0.965812
4051,Ty Johnson,BUF,17,24.069822,1.415872,1.733613,1.498905,1.733613,6.0,0.957265
4161,James Cook,BUF,16,23.395136,1.462196,1.692724,1.535839,1.692724,7.0,0.948718
3979,Austin Ekeler,WAS,12,18.801500,1.566792,1.414334,1.619231,1.414334,8.0,0.940171
4261,Sean Tucker,TB,17,17.547465,1.032204,1.338336,1.193012,1.338336,9.0,0.931624
4032,Justice Hill,BAL,15,14.844525,0.989635,1.174528,1.159073,1.174528,10.0,0.923077



Top WRs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
4140,Amon-Ra St. Brown,DET,17,95.743892,5.631994,4.230455,3.223783,4.230455,1.0,1.000000
4133,Ja'Marr Chase,CIN,17,79.032173,4.648951,3.388240,2.564870,3.388240,2.0,0.995122
4344,Ladd McConkey,LAC,16,65.707443,4.106715,2.716718,2.201421,2.716718,3.0,0.990244
4060,Terry McLaurin,WAS,17,64.637547,3.802209,2.662798,1.997316,2.662798,4.0,0.985366
4089,Justin Jefferson,MIN,17,63.974392,3.763200,2.629377,1.971169,2.629377,5.0,0.980488
4061,A.J. Brown,PHI,13,61.811166,4.754705,2.520358,2.635755,2.520358,6.0,0.975610
4339,Brian Thomas Jr.,JAX,17,56.810445,3.341791,2.268338,1.688708,2.268338,7.0,0.970732
4097,Tee Higgins,CIN,12,56.401296,4.700108,2.247719,2.599159,2.247719,8.0,0.965854
4136,DeVonta Smith,PHI,13,56.324606,4.332662,2.243854,2.352868,2.243854,9.0,0.960976
4281,Puka Nacua,LA,11,54.491021,4.953729,2.151447,2.769156,2.151447,10.0,0.956098



Top TEs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
3965,George Kittle,SF,15,67.944928,4.529662,4.122215,4.028512,4.122215,1.0,1.000000
3982,Jonnu Smith,MIA,17,58.068716,3.415807,3.436751,2.892112,3.436751,2.0,0.990291
4017,Mark Andrews,BAL,17,53.545150,3.149715,3.122790,2.620634,3.122790,3.0,0.980583
4191,Trey McBride,ARI,16,49.672450,3.104528,2.854003,2.574532,2.854003,4.0,0.970874
4266,Tucker Kraft,GB,17,40.513069,2.383122,2.218290,1.838524,2.218290,5.0,0.961165
4132,Pat Freiermuth,PIT,17,35.901233,2.111837,1.898203,1.561748,1.898203,6.0,0.951456
4023,Mike Gesicki,CIN,16,33.767575,2.110473,1.750115,1.560357,1.750115,7.0,0.941748
4277,Sam LaPorta,DET,16,31.423357,1.963960,1.587413,1.410878,1.587413,8.0,0.932039
3926,Zach Ertz,WAS,17,30.982541,1.822502,1.556818,1.266557,1.556818,9.0,0.922330
4198,Isaiah Likely,BAL,15,27.706500,1.847100,1.329442,1.291653,1.329442,10.0,0.912621


## Inspect Bottom Rankings

I also inspect the lowest-ranked players to check whether the bottom of the distribution is reasonable. This can reveal issues with missing data, low-volume players, or unusual position assignments.

In [9]:
season_to_check = 2024

for pos in ["QB", "RB", "WR", "TE"]:
    print()
    print("Bottom", pos + "s", "in", season_to_check)
    display(
        value_scored[
            (value_scored["season"] == season_to_check) &
            (value_scored["position"] == pos)
        ][
            ["player_display_name", "team", "games_played",
             "value_epa_total", "value_epa_per_game",
             "value_score_total_epa", "value_score_per_game",
             "value_score", "position_season_rank",
             "position_season_percentile"]
        ]
        .sort_values("value_score", ascending=True)
        .head(10)
    )



Bottom QBs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
478,Will Levis,TEN,12,-83.015486,-6.917957,-1.662888,-1.560842,-1.662888,58.0,0.017241
475,Dorian Thompson-Robinson,CLE,6,-65.149749,-10.858292,-1.370827,-2.386878,-1.370827,57.0,0.034483
444,Deshaun Watson,CLE,7,-63.386317,-9.055188,-1.342000,-2.008883,-1.342000,56.0,0.051724
481,Spencer Rattler,NO,7,-62.163924,-8.880561,-1.322017,-1.972275,-1.322017,55.0,0.068966
454,Gardner Minshew,LV,10,-53.575489,-5.357549,-1.181617,-1.233724,-1.181617,54.0,0.086207
445,Cooper Rush,DAL,12,-41.031081,-3.419257,-0.976547,-0.827388,-0.976547,53.0,0.103448
486,Caleb Williams,CHI,17,-35.642760,-2.096633,-0.888461,-0.550118,-0.888461,52.0,0.120690
442,Jacoby Brissett,NE,7,-33.998105,-4.856872,-0.861575,-1.128764,-0.861575,51.0,0.137931
456,Daniel Jones,NYG,10,-30.137593,-3.013759,-0.798465,-0.742381,-0.798465,50.0,0.155172
455,Drew Lock,NYG,7,-29.594010,-4.227716,-0.789579,-0.996870,-0.789579,49.0,0.172414



Bottom RBs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
4043,Tony Pollard,TEN,16,-40.615764,-2.538485,-2.186555,-1.653843,-2.186555,117.0,0.008547
4200,Zamir White,LV,8,-38.537334,-4.817167,-2.060595,-3.470601,-2.060595,116.0,0.017094
4031,Alexander Mattison,LV,14,-37.063236,-2.647374,-1.971260,-1.740658,-1.971260,115.0,0.025641
4020,Nick Chubb,CLE,8,-36.891935,-4.611492,-1.960879,-3.306619,-1.960879,114.0,0.034188
4086,D'Andre Swift,CHI,17,-36.066499,-2.121559,-1.910854,-1.321434,-1.910854,113.0,0.042735
4077,Jonathan Taylor,IND,14,-34.645792,-2.474699,-1.824755,-1.602988,-1.824755,112.0,0.051282
4142,Travis Etienne,JAX,15,-31.505121,-2.100341,-1.634419,-1.304518,-1.634419,111.0,0.059829
4126,Rhamondre Stevenson,NE,15,-30.470066,-2.031338,-1.571691,-1.249502,-1.571691,110.0,0.068376
4146,Javonte Williams,DEN,17,-29.431437,-1.731261,-1.508747,-1.010256,-1.508747,109.0,0.076923
4301,Tyrone Tracy Jr.,NYG,17,-26.583517,-1.563736,-1.336153,-0.876691,-1.336153,108.0,0.085470



Bottom WRs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
4143,Elijah Moore,CLE,17,-37.346487,-2.196852,-2.476859,-2.023729,-2.476859,205.0,0.004878
4206,Wan'Dale Robinson,NYG,17,-26.920628,-1.583566,-1.951430,-1.612656,-1.951430,204.0,0.009756
4342,Ja'Lynn Polk,NE,14,-21.436259,-1.531161,-1.675036,-1.577530,-1.675036,203.0,0.014634
4334,Troy Franklin,DEN,16,-18.681420,-1.167589,-1.536201,-1.333835,-1.536201,202.0,0.019512
4241,Jalen Brooks,DAL,12,-17.675178,-1.472931,-1.485490,-1.538500,-1.485490,201.0,0.024390
3934,Brandin Cooks,DAL,10,-13.943234,-1.394323,-1.297412,-1.485811,-1.297412,200.0,0.029268
4338,Adonai Mitchell,IND,17,-13.740462,-0.808262,-1.287193,-1.092986,-1.287193,199.0,0.034146
3964,Curtis Samuel,BUF,14,-12.188647,-0.870618,-1.208987,-1.134782,-1.208987,198.0,0.039024
3933,Odell Beckham Jr.,MIA,8,-11.280473,-1.410059,-1.163218,-1.496358,-1.163218,197.0,0.043902
4274,Jonathan Mingo,CAR,8,-10.908070,-1.363509,-1.144450,-1.465156,-1.144450,196.0,0.048780



Bottom TEs in 2024


,player_display_name,team,games_played,value_epa_total,value_epa_per_game,value_score_total_epa,value_score_per_game,value_score,position_season_rank,position_season_percentile
4201,Jake Ferguson,DAL,14,-12.711065,-0.907933,-1.475765,-1.519146,-1.475765,103.0,0.009709
4127,Kylen Granson,IND,16,-8.418040,-0.526128,-1.177804,-1.129612,-1.177804,102.0,0.019417
3942,MyCole Pruitt,PIT,7,-8.000600,-1.142943,-1.148832,-1.758912,-1.148832,101.0,0.029126
3970,Pharaoh Brown,SEA,10,-6.471276,-0.647128,-1.042688,-1.253061,-1.042688,100.0,0.038835
4195,Chig Okonkwo,TEN,16,-6.252948,-0.390809,-1.027535,-0.991554,-1.027535,99.0,0.048544
4213,Will Mallory,IND,7,-5.917483,-0.845355,-1.004252,-1.455300,-1.004252,98.0,0.058252
4104,Charlie Woerner,ATL,12,-5.707398,-0.475616,-0.989671,-1.078078,-0.989671,97.0,0.067961
3984,David Njoku,CLE,11,-5.206480,-0.473316,-0.954904,-1.075732,-0.954904,96.0,0.077670
4024,Hayden Hurst,LAC,7,-4.841557,-0.691651,-0.929576,-1.298486,-0.929576,95.0,0.087379
4283,Luke Musgrave,GB,7,-4.797619,-0.685374,-0.926527,-1.292082,-0.926527,94.0,0.097087


## Save Player Value Scores

This saves the value-scored player-season dataset locally for exploratory analysis, predictive modeling, and later salary-efficiency analysis. The processed CSV is intentionally ignored by Git.

In [10]:
output_path = processed_dir / "player_value_scores_2016_2025.csv"

value_scored.to_csv(output_path, index=False)

print(f"Saved {len(value_scored):,} rows to {output_path}")


Saved 4,796 rows to /Users/kylelevesque/Desktop/nfl-player-value-analysis-1/data/processed/player_value_scores_2016_2025.csv
